In [0]:
from pyspark.sql import SparkSession

spark.sql("CREATE DATABASE IF NOT EXISTS flight_data_silver")
spark.sql("USE flight_data_silver")

DataFrame[]

In [0]:
%sql
CREATE TABLE IF NOT EXISTS flight_data_silver.dimairports(
  airport_id string,
  airport_name string,
  airport_name_OLD string,
  airport_name_effective_date TIMESTAMP,
  city string,
  state string,
  country string,
  latitude double,
  longitude double
)

In [0]:
%sql

MERGE INTO flight_data_silver.dimairports AS dima
USING flight_data_bronze.airports AS a
ON dima.airport_id = a.airport_id
WHEN MATCHED AND(
    dima.airport_name <> a.airport_name OR
    dima.city <> a.city OR
    dima.state <> a.state OR
    dima.country <> a.country OR
    dima.latitude <> a.latitude OR
    dima.longitude <> a.longitude
) THEN
UPDATE SET
    dima.airport_name_OLD = (CASE WHEN dima.airport_name <> a.airport_name
                             THEN dima.airport_name
                             ELSE dima.airport_name_OLD END),
    dima.airport_name = a.airport_name,
    dima.airport_name_effective_date = (CASE WHEN dima.airport_name <> a.airport_name
                                        THEN current_timestamp()
                                        ELSE dima.airport_name_effective_date END),
    dima.city = a.city,
    dima.state = a.state,
    dima.country = a.country,
    dima.latitude = a.latitude,
    dima.longitude = a.longitude
WHEN NOT MATCHED
THEN INSERT(
    airport_id, airport_name, airport_name_OLD, airport_name_effective_date, city, state, country, latitude, longitude)
VALUES(
    a.airport_id, a.airport_name, NULL, current_timestamp(), a.city, a.state, a.country, a.latitude, a.longitude)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
322,0,0,322


In [0]:
%sql
SELECT * FROM flight_data_silver.dimairports

airport_id,airport_name,airport_name_OLD,airport_name_effective_date,city,state,country,latitude,longitude
ABE,Lehigh Valley International Airport,null,2025-12-12T11:27:38.996Z,Allentown,PA,USA,40.65235900878906,-75.44039916992188
ABI,Abilene Regional Airport,null,2025-12-12T11:27:38.996Z,Abilene,TX,USA,32.411319732666016,-99.68190002441406
ABQ,Albuquerque International Sunport,null,2025-12-12T11:27:38.996Z,Albuquerque,NM,USA,35.040218353271484,-106.60919189453125
ABR,Aberdeen Regional Airport,null,2025-12-12T11:27:38.996Z,Aberdeen,SD,USA,45.449058532714844,-98.42182922363281
ABY,Southwest Georgia Regional Airport,null,2025-12-12T11:27:38.996Z,Albany,GA,USA,31.535520553588867,-84.19447326660156
ACK,Nantucket Memorial Airport,null,2025-12-12T11:27:38.996Z,Nantucket,MA,USA,41.2530517578125,-70.0601806640625
ACT,Waco Regional Airport,null,2025-12-12T11:27:38.996Z,Waco,TX,USA,31.611289978027344,-97.23052215576172
ACV,Arcata Airport,null,2025-12-12T11:27:38.996Z,Arcata/Eureka,CA,USA,40.978118896484375,-124.1086196899414
ACY,Atlantic City International Airport,null,2025-12-12T11:27:38.996Z,Atlantic City,NJ,USA,39.45758056640625,-74.5771713256836
ADK,Adak Airport,null,2025-12-12T11:27:38.996Z,Adak,AK,USA,51.877960205078125,-176.64602661132812
